In [31]:
import os
import json
import logging
from difflib import get_close_matches

import requests
import gradio as gr
from dotenv import load_dotenv
from openai import (
    OpenAI,
    APIConnectionError,
    APIError,
    APITimeoutError,
    BadRequestError,
    RateLimitError,
)

from content_extractor import extract_text

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("restaurant-bot")


In [32]:
load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [33]:
if GROQ_API_KEY is None or GROQ_API_KEY.strip() == "":
    raise ValueError("GROQ_API_KEY is not set in the environment variables.")
else:
    print("Groq API Key found")

Groq API Key found


In [34]:
MODEL = "openai/gpt-oss-120b"

REQUEST_TIMEOUT = 30.0        # seconds before a single API call is abandoned
SDK_MAX_RETRIES = 3           # SDK retries 429/5xx with backoff for us
MAX_TOOL_ROUNDS = 4           # safety net; a normal answer needs 2 rounds
MAX_COMPLETION_TOKENS = 800   # caps a runaway generation (and its cost)
MAX_HISTORY_MESSAGES = 12     # ~6 turns; keeps prompt size flat instead of growing
MAX_ITEMS_PER_CALL = 25       # guards against the model sending a huge array

openai = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    timeout=REQUEST_TIMEOUT,
    max_retries=SDK_MAX_RETRIES,
)


In [35]:
SYSTEM_PROMPT = """
You are a snarky assistant who owns the restaurant.

Rules:
- When the user asks for the menu, available food, available dishes, or what is in stock, ALWAYS call the getFoodNames tool.
- Never guess or invent menu items.
- When the user asks about prices, ALWAYS call the getPriceOfFood tool.
- If the user asks about several items at once, pass ALL of them in ONE getPriceOfFood call.
  Never make a separate call per item.
- Never guess or invent prices.
- If you don't know something and no tool can provide the answer, say: "I don't know about it."
- After receiving tool results, answer the user naturally in one short reply.
"""


In [36]:
url = "https://migrationology.com/pakistani-street-food-karachi/"

# Network fetch: never let a blip here stop the rest of the notebook from running.
try:
    responseText = extract_text(url)
except Exception:
    logger.exception("Could not fetch %s", url)
    responseText = ""


In [37]:
USER_PROMPT = f"Summarize the following text:' {responseText}'"

In [38]:
foods = [
    {
        "name": "Nihari",
        "category": "Breakfast",
        "price": 700
    },
    {
        "name": "Arabian Paratha",
        "category": "Breakfast",
        "price": 450
    },
    {
        "name": "Kulfi (Clay Cup)",
        "category": "Dessert",
        "price": 250
    },
    {
        "name": "Rabri",
        "category": "Dessert",
        "price": 350
    },
    {
        "name": "Bone Marrow Biryani",
        "category": "Main Course",
        "price": 1200
    },
    {
        "name": "Ninja Street Salad",
        "category": "Street Food",
        "price": 300
    },
    {
        "name": "Fish Kata-Kat",
        "category": "Main Course",
        "price": 900
    },
    {
        "name": "Bun Kebab",
        "category": "Street Food",
        "price": 180
    },
    {
        "name": "Chicken Curry",
        "category": "Main Course",
        "price": 650
    }
]

In [39]:
def _normalize(name):
    """Lowercase, trim, and collapse inner whitespace so lookups are forgiving."""
    return " ".join(str(name).strip().lower().split())


# Built once: turns each price lookup into a dict hit instead of a list scan.
FOOD_BY_NAME = {_normalize(food["name"]): food for food in foods}

if len(FOOD_BY_NAME) != len(foods):
    raise ValueError("Duplicate food names in `foods`; names must be unique.")


def getPriceOfFood(food_name):
    key = _normalize(food_name)
    if not key:
        return "No food name was given."

    food = FOOD_BY_NAME.get(key)
    if food is None:
        # Tolerate near-misses: "kulfi" -> "Kulfi (Clay Cup)", "nihri" -> "Nihari".
        matches = [n for n in FOOD_BY_NAME if key in n]
        if not matches:
            matches = get_close_matches(key, list(FOOD_BY_NAME), n=1, cutoff=0.8)
        if len(matches) == 1:
            food = FOOD_BY_NAME[matches[0]]

    if food is None:
        return f"{food_name} is not on the menu."
    return f"the price of {food['name']} is {food['price']:,} PKR."


def getPricesOfFoods(food_names):
    """Price several items in one go, so N items cost one tool round, not N."""
    seen = set()
    unique = []
    for name in food_names[:MAX_ITEMS_PER_CALL]:
        if not isinstance(name, str):
            continue  # the model can emit null or a number inside the array
        key = _normalize(name)
        if key and key not in seen:
            seen.add(key)
            unique.append(name)

    if not unique:
        return "No food items were given."
    return "\n".join(getPriceOfFood(name) for name in unique)


In [40]:
def getFoodNames(foods=foods):
    return [food["name"] for food in foods]


In [41]:
price_function = {
    "name": "getPriceOfFood",
    "description": (
        "Get the price of one or more food items from the menu. "
        "Pass every item the user asked about in a single call."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "food_names": {
                "type": "array",
                "items": {"type": "string"},
                "description": "The names of all food items to get prices for."
            }
        },
        "required": ["food_names"],
        "additionalProperties": False
    }
}

get_food_details = {
    "name": "getFoodNames",
    "description": "Get all the available food names currently on the menu.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}


In [42]:
tools = [
    {
        "type": "function",
        "function": price_function
    },
    {
        "type": "function",
        "function": get_food_details
    }
]

In [43]:
def _tool_get_prices(arguments):
    food_names = arguments.get("food_names")
    if food_names is None:
        # The model sometimes ignores the schema and sends the old singular key.
        single = arguments.get("food_name")
        food_names = [single] if single else []
    if isinstance(food_names, str):
        food_names = [food_names]
    if not isinstance(food_names, list):
        return "Invalid food_names argument; expected a list of names."
    return getPricesOfFoods(food_names)


def _tool_get_names(arguments):
    return ", ".join(getFoodNames())


TOOL_REGISTRY = {
    "getPriceOfFood": _tool_get_prices,
    "getFoodNames": _tool_get_names,
}


def handle_tool_call(message):
    """Run every tool the model asked for. Always returns one result per call id."""
    results = []
    for tool_call in message.tool_calls or []:
        name = tool_call.function.name
        try:
            arguments = json.loads(tool_call.function.arguments or "{}")
            if not isinstance(arguments, dict):
                raise ValueError("arguments must be a JSON object")
        except (json.JSONDecodeError, ValueError) as exc:
            logger.warning("Bad arguments for %s: %s", name, exc)
            result = f"Invalid arguments for {name}."
        else:
            handler = TOOL_REGISTRY.get(name)
            if handler is None:
                logger.warning("Model asked for unknown tool: %s", name)
                result = f"Unknown tool: {name}."
            else:
                try:
                    result = handler(arguments)
                except Exception:
                    # A tool bug must not take down the chat.
                    logger.exception("Tool %s raised", name)
                    result = f"{name} failed."

        results.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id,
        })
    return results


In [44]:
FALLBACK_REPLY = "Sorry, my kitchen is a mess right now. Try that again in a moment."


def _clean_history(history):
    """Gradio can hand back file turns or None; keep only well-formed text turns."""
    cleaned = []
    for turn in history or []:
        if not isinstance(turn, dict):
            continue
        role = turn.get("role")
        content = turn.get("content")
        if role in ("user", "assistant") and isinstance(content, str) and content.strip():
            cleaned.append({"role": role, "content": content})
    # Trim to the recent window so prompt cost stays flat as the chat grows.
    return cleaned[-MAX_HISTORY_MESSAGES:]


def chat(message, history):
    if not isinstance(message, str) or not message.strip():
        return "Say something and I will tell you what we have got."

    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + _clean_history(history)
        + [{"role": "user", "content": message.strip()}]
    )

    for round_index in range(MAX_TOOL_ROUNDS):
        # Drop the tools on the final round so the model has to answer in words.
        offer_tools = round_index < MAX_TOOL_ROUNDS - 1
        request = {
            "model": MODEL,
            "messages": messages,
            "max_tokens": MAX_COMPLETION_TOKENS,
        }
        if offer_tools:
            request["tools"] = tools

        try:
            response = openai.chat.completions.create(**request)
        except (APITimeoutError, APIConnectionError):
            logger.exception("Groq unreachable")
            return "I cannot reach the kitchen right now. Try again in a moment."
        except RateLimitError:
            logger.exception("Rate limited")
            return "Too many orders at once. Give me a few seconds and ask again."
        except BadRequestError:
            logger.exception("Rejected request")
            return FALLBACK_REPLY
        except APIError:
            logger.exception("Groq API error")
            return FALLBACK_REPLY

        choice = response.choices[0]
        reply = choice.message

        if choice.finish_reason != "tool_calls" or not reply.tool_calls:
            return (reply.content or "").strip() or "I don't know about it."

        messages.append({
            "role": "assistant",
            "content": reply.content or "",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
                for tool_call in reply.tool_calls
            ]
        })
        messages.extend(handle_tool_call(reply))

    return FALLBACK_REPLY


In [45]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Restaurant Assistant",
    examples=["What is on the menu?", "Price of Bun Kebab and Chicken Curry"],
).queue(max_size=32).launch()


2026-09-02 20:24:05,665 INFO HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-09-02 20:24:05,763 INFO HTTP Request: GET http://127.0.0.1:7862/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-09-02 20:24:05,798 INFO HTTP Request: HEAD http://127.0.0.1:7862/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


2026-09-02 20:24:06,034 INFO HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-09-02 20:24:06,893 INFO HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-09-02 20:24:14,784 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-02 20:24:15,344 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-02 20:24:37,528 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-02 20:24:37,705 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-02 20:24:58,607 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-02 20:24:59,413 INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-02 20:25:43,800 INFO HTTP Request: POST https://api.gro